# NB-09 — T5 end-to-end: validated plan → XLSX artifact → recorded fills

The composition notebook. Ships alongside T5 phases P1 (#1749), P2
(#1752), and P3.a (#1755) of #1719. Assumes you've skimmed:

- **NB-04** for how a plan gets validated
- **NB-07** for OrderBatch + the 6-sheet XLSX artifact
- **NB-08** for the paper trading engine's ledger discipline

This notebook is the story that pulls everything together: from a
validated plan through the artifact you file at Fidelity, through
the paper engine that shadows what actually filled.

## The full T5 flow

```
   T4 validated plan
         │
         ▼
   OrderBatch (composed with tickets + verdict gates + pricing + snapshot)
         │
         ▼
   PaperOrderSink.write_batch
    ├── CSV (Fidelity Basket Trading format — reference/portability only)
    └── XLSX (6-sheet review workbook — the operator reads THIS before filing)
         │
         ▼
   Operator files each order manually at Fidelity's UI
         │
         ▼
   Fidelity confirms fills (email / activity page / paper printout)
         │
         ▼
   SqlitePaperEngine.record_fill (one call per fill Fidelity confirms)
    ├── cash side effects
    ├── position derivation
    └── FIFO realized P&L
         │
         ▼
   Operator's local book stays aligned with Fidelity's real book
```

## What this notebook does NOT cover

- Unrealized P&L (P3.b, not shipped)
- Reconciliation vs. Fidelity's real positions snapshot (P3.c, not shipped)
- The `tt_execute_bridge` widget rewrite (P4, not shipped)
- Activity CSV bulk-import parsing (P5, not shipped)

Full test guide at `docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md`.


## 1. The validated plan (from T4)

In real usage the plan comes out of `nb-04-validation-gate` with all
its verdict gates already computed. Here we mimic that with a synthetic
plan — same shape, no external dependencies.


In [ ]:
from decimal import Decimal

# What T4 would emit — normally imported from the plan store.
validated_plan = {
    "plan_id": "nb09-morning-2026-08-03",
    "verdict_gate_pass": True,
    "generator_version": "techtrade-plan-1.5.0",
    "git_sha": "0000000000000000000000000000000000000000",
    "verdict_gates": [
        {"name": "max_single_position", "threshold": "0.10", "actual": "0.08", "passed": True},
        {"name": "max_sector_exposure", "threshold": "0.30", "actual": "0.28", "passed": True},
        {"name": "min_liquidity_adv",   "threshold": "1M",   "actual": "12M",  "passed": True},
    ],
    "orders": [
        {"symbol": "MSFT", "action": "Buy",  "quantity": "50",  "order_type": "Limit",  "limit_price": "400.00"},
        {"symbol": "AAPL", "action": "Buy",  "quantity": "100", "order_type": "Limit",  "limit_price": "180.00"},
        {"symbol": "NVDA", "action": "Sell", "quantity": "25",  "order_type": "Limit",  "limit_price": "130.00"},
    ],
    "pricing": {"MSFT": "398.00", "AAPL": "179.50", "NVDA": "128.00"},
    "pre_execution_positions": {"MSFT": "0.05", "AAPL": "0.03", "NVDA": "0.08"},
}
print(f"Plan ID:       {validated_plan['plan_id']}")
print(f"Verdict:       {'PASS' if validated_plan['verdict_gate_pass'] else 'FAIL'}")
print(f"# gates:       {len(validated_plan['verdict_gates'])} (all passed)")
print(f"# orders:      {len(validated_plan['orders'])}")

Plan ID:       nb09-morning-2026-08-03
Verdict:       PASS
# gates:       3 (all passed)
# orders:      3


## 2. Compose the OrderBatch

Translate the plan into the frozen dataclasses `OrderBatch` accepts.
This is the seam between T4 (validation) and T5 (execute).

For deeper walkthroughs of OrderTicket construction and the SHA
idempotency invariant, see NB-07.


In [ ]:
from openbb_techtrade.execution.order_sink import (
    OrderBatch, OrderTicket, PlanContext, VerdictGate,
)

tickets = tuple(
    OrderTicket(
        symbol=o["symbol"],
        action=o["action"],
        quantity=Decimal(o["quantity"]),
        order_type=o["order_type"],
        limit_price=Decimal(o["limit_price"]) if o.get("limit_price") else None,
        account_masked="***1234",
    )
    for o in validated_plan["orders"]
)

verdict_gates = tuple(
    VerdictGate(name=g["name"], threshold=g["threshold"],
                actual=g["actual"], passed=g["passed"])
    for g in validated_plan["verdict_gates"]
)

batch = OrderBatch(
    tickets=tickets,
    plan_id=validated_plan["plan_id"],
    verdict_gate_pass=validated_plan["verdict_gate_pass"],
    pricing={k: Decimal(v) for k, v in validated_plan["pricing"].items()},
    pre_execution_positions={k: Decimal(v) for k, v in validated_plan["pre_execution_positions"].items()},
    plan_context=PlanContext(
        verdict_gates=verdict_gates,
        generator_version=validated_plan["generator_version"],
        git_sha=validated_plan["git_sha"],
    ),
)

print(f"Batch composed with {len(batch.tickets)} tickets")
print(f"Batch SHA (short): {batch.sha_short()}")

Batch composed with 3 tickets
Batch SHA (short): 6f368055


## 3. Write the artifact

`PaperOrderSink` writes CSV + XLSX atomically. The XLSX is what the
operator reads before typing orders at Fidelity. Filename embeds
the batch SHA so a shared drive can match paired files trivially.


In [ ]:
import tempfile
from pathlib import Path
from openbb_techtrade.execution.order_sink import PaperOrderSink

out_dir = Path(tempfile.mkdtemp(prefix="nb09_exec_"))
sink = PaperOrderSink(out_dir)
art = sink.write_batch(batch)

print(f"CSV path:  ...{art.csv_path.name}")
print(f"XLSX path: ...{art.xlsx_path.name}")
print(f"SHA (full): {art.batch_sha256}")

CSV path:  ...2026-08-03-6f368055.csv
XLSX path: ...2026-08-03-6f368055.xlsx
SHA (full): 6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a


### Quick verification: the artifact contains what T4 said

Full walkthrough of the 6 sheets is in NB-07 — here we just confirm
the verdict gates rendered (Plan Context) and the SHA + provenance
made it into the Audit sheet.


In [ ]:
from openpyxl import load_workbook

wb = load_workbook(art.xlsx_path)
print(f"Sheets in workbook: {wb.sheetnames}")

# Plan Context — verify all 3 T4 gates rendered.
print("\nPlan Context sheet (all gates should show PASS):")
for row in wb["Plan Context"].iter_rows(values_only=True):
    print(f"  {row}")

# Audit — verify SHA + git provenance landed.
print("\nAudit sheet (provenance):")
for row in wb["Audit"].iter_rows(values_only=True):
    if row[0] in ("Batch SHA (full)", "Plan ID", "Generator version", "Git SHA"):
        print(f"  {row[0]:<24} {row[1]}")

Sheets in workbook: ['Orders', 'Batch Summary', 'Plan Context', 'Deviation Analysis', 'Concentration', 'Audit']

Plan Context sheet (all gates should show PASS):
  ('Gate', 'Threshold', 'Actual', 'Passed', 'Notes')
  ('max_single_position', '0.10', '0.08', 'PASS', None)
  ('max_sector_exposure', '0.30', '0.28', 'PASS', None)
  ('min_liquidity_adv', '1M', '12M', 'PASS', None)

Audit sheet (provenance):
  Batch SHA (full)         6f368055d33ebb327cd95db249af3c636e5d112ef1cc8ae38dd4677d7f56bd7a
  Plan ID                  nb09-morning-2026-08-03
  Generator version        techtrade-plan-1.5.0
  Git SHA                  0000000000000000000000000000000000000000


## 4. Print the operator's "type this at Fidelity" cheat sheet

Even though we don't upload to Fidelity, we print the CSV rows in
the shape the operator types into Fidelity's order-entry UI. Each
row is one order the operator will place manually.


In [ ]:
print("Operator: type these into Fidelity, one at a time.")
print("-" * 70)
print(f"{'Symbol':<6}  {'Side':<5}  {'Qty':>5}  {'Type':<8}  {'Limit':>8}  {'TIF':<4}")
print("-" * 70)
for t in batch.tickets:
    limit = f"{t.limit_price}" if t.limit_price is not None else "—"
    print(f"{t.symbol:<6}  {t.action:<5}  {t.quantity:>5}  {t.order_type:<8}  {limit:>8}  {t.tif:<4}")

Operator: type these into Fidelity, one at a time.
----------------------------------------------------------------------
Symbol  Side     Qty  Type         Limit  TIF 
----------------------------------------------------------------------
MSFT    Buy       50  Limit       400.00  Day 
AAPL    Buy      100  Limit       180.00  Day 
NVDA    Sell      25  Limit       130.00  Day 


## 5. Submit the batch to the paper engine

The paper engine records each ticket as a PENDING order. The operator
uses the returned `order_id` values later when recording fills.

Deep-dive on the engine's data model is in NB-08.


In [ ]:
from openbb_techtrade.execution.paper_engine import SqlitePaperEngine

paper_dir = Path(tempfile.mkdtemp(prefix="nb09_paper_"))
engine = SqlitePaperEngine(paper_dir / "paper.db",
                            starting_cash=Decimal("100000"))

order_ids = engine.submit_batch(batch, plan_id=batch.plan_id)
print(f"Submitted {len(order_ids)} orders to the paper engine.")
print(f"Starting cash: {engine.get_account().cash}")

# Map order_id → symbol for readability below.
by_symbol = {}
for oid in order_ids:
    for o in engine.get_orders():
        if o.order_id == oid:
            by_symbol[o.symbol] = oid
print(f"\nOrder IDs by symbol:")
for sym, oid in by_symbol.items():
    print(f"  {sym}: {oid}")

Submitted 3 orders to the paper engine.
Starting cash: 100000

Order IDs by symbol:
  MSFT: ord_985a4ddb60ab
  AAPL: ord_41725cd1b86b
  NVDA: ord_7b2dad4bfc35


## 6. Simulate the operator's Fidelity workflow

The operator files each order and gets a confirmation. For this
demo we imagine:

1. **MSFT** — filled 50 @ $398.50 with $1 commission (matched the limit)
2. **AAPL** — filled 100 @ $179.90 with $1 commission (crossed above limit — filled anyway)
3. **NVDA** — didn't fill at $130, operator cancels after end of day


In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)

# 1. MSFT — full fill.
engine.record_fill(by_symbol["MSFT"], price=Decimal("398.50"),
                   filled_qty=Decimal("50"), at=now, commission=Decimal("1"))
print(f"MSFT filled: 50 @ 398.50. Cash: {engine.get_account().cash}")

# 2. AAPL — full fill at slightly worse price.
engine.record_fill(by_symbol["AAPL"], price=Decimal("179.90"),
                   filled_qty=Decimal("100"), at=now, commission=Decimal("1"))
print(f"AAPL filled: 100 @ 179.90. Cash: {engine.get_account().cash}")

# 3. NVDA — didn't fill; cancel.
engine.cancel_order(by_symbol["NVDA"], reason="No fill at $130 EOD")
print(f"NVDA cancelled")

print("\n--- End-of-day state ---")
for o in engine.get_orders():
    print(f"  {o.symbol:6} {o.status.value}")

MSFT filled: 50 @ 398.50. Cash: 80074.00
AAPL filled: 100 @ 179.90. Cash: 62083.00
NVDA cancelled

--- End-of-day state ---
  MSFT   FILLED
  AAPL   FILLED
  NVDA   CANCELLED


## 7. Position + P&L after the fills

Every buy that filled shows up as a long position at the actual
execution price (not the limit). Realized P&L is 0 so far — nothing
has been closed. Cash reflects the total spent + commissions.


In [ ]:
acct = engine.get_account()
print(f"Account cash:       {acct.cash}")
print(f"Realized P&L:       {acct.realized_pl}")
print(f"\nPositions:")
for p in engine.get_positions():
    notional_paid = p.quantity * p.avg_cost
    print(f"  {p.symbol:6} qty={p.quantity} avg_cost={p.avg_cost} → paid {notional_paid}")

# Sanity check the cash arithmetic:
# 100_000 - (50 * 398.50 + 1) - (100 * 179.90 + 1) = 100_000 - 19_926 - 17_991 = 62_083.
print(f"\nCash check: 100_000 - 19_926 - 17_991 = 62_083 (matches account: {acct.cash})")

Account cash:       62083.00
Realized P&L:       0

Positions:
  AAPL   qty=100 avg_cost=179.90 → paid 17990.00
  MSFT   qty=50 avg_cost=398.50 → paid 19925.00

Cash check: 100_000 - 19_926 - 17_991 = 62_083 (matches account: 62083.00)


## 8. Partial-close scenario — take profit on MSFT

A day later MSFT has run up. The operator files a sell of 30 shares
at $410 at Fidelity. Fidelity fills it; the operator records the
fill in the paper engine. The FIFO ledger books the realized P&L.

(Deep-dive on FIFO math is in NB-08 §7.)


In [ ]:
[profit_id] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="MSFT", action="Sell", quantity=Decimal("30"),
                order_type="Market"),
)))
engine.record_fill(profit_id, price=Decimal("410.00"),
                   filled_qty=Decimal("30"), at=now, commission=Decimal("1"))

# Expected realized: 30 * (410 - 398.50) = 30 * 11.50 = 345.
acct = engine.get_account()
[msft_pos] = [p for p in engine.get_positions() if p.symbol == "MSFT"]
print(f"Realized P&L on the close: 30 * (410 - 398.50) = 345 (matches: {msft_pos.realized_pl})")
print(f"Account realized P&L: {acct.realized_pl}")
print(f"\nMSFT position after partial close: {msft_pos.quantity} shares @ avg_cost {msft_pos.avg_cost}")
print(f"(avg_cost unchanged on partial close — FIFO already booked the realized P&L)")

Realized P&L on the close: 30 * (410 - 398.50) = 345 (matches: 345.00)
Account realized P&L: 345.00

MSFT position after partial close: 20 shares @ avg_cost 398.50
(avg_cost unchanged on partial close — FIFO already booked the realized P&L)


## 9. End-of-day review

The operator now has a clean end-of-day picture. Every position is
tied back to its plan via `batch_sha256` on the order rows. The audit
sheet on the original XLSX still carries the plan provenance for
post-facto reconciliation months later.


In [ ]:
print("=" * 60)
print("END-OF-DAY REVIEW")
print("=" * 60)
acct = engine.get_account()
print(f"Cash:              {acct.cash}")
print(f"Realized P&L:      {acct.realized_pl}")
print()
print(f"Open positions:")
for p in engine.get_positions():
    print(f"  {p.symbol:6} {p.quantity:>6} shares @ avg_cost {p.avg_cost}")
print()
print(f"Order history (this session):")
for o in engine.get_orders():
    print(f"  {o.symbol:6} {o.side.value:5} qty={o.quantity} status={o.status.value}")
print()
print(f"Fill history:")
for f in engine.get_fills():
    print(f"  {f.symbol:6} {f.side.value:5} {f.filled_qty} @ {f.price} (commission {f.commission})")

engine.close()

END-OF-DAY REVIEW
Cash:              74382.00
Realized P&L:      345.00

Open positions:
  AAPL      100 shares @ avg_cost 179.90
  MSFT       20 shares @ avg_cost 398.50

Order history (this session):
  MSFT   BUY   qty=50 status=FILLED
  AAPL   BUY   qty=100 status=FILLED
  NVDA   SELL  qty=25 status=CANCELLED
  MSFT   SELL  qty=30 status=FILLED

Fill history:
  MSFT   BUY   50 @ 398.50 (commission 1)
  AAPL   BUY   100 @ 179.90 (commission 1)
  MSFT   SELL  30 @ 410.00 (commission 1)


## 10. What's next

- **NB-07** — deeper dive on OrderTicket / OrderBatch / XLSX shape
- **NB-08** — deeper dive on the paper trading engine (FIFO, cancels, loud rejections)
- **Test guide** — `docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md` — QA + automation scenarios
- **Fidelity CSV schema** — `docs/superpowers/specs/2026-08-03-fidelity-positions-csv-schema.md`

Unshipped follow-ons on #1719:
- **P3.b** — unrealized P&L (needs live prices)
- **P3.c** — reconciliation vs. Fidelity positions snapshot
- **P4** — widget rewrite (write-batch button + double-gate)
- **P5** — bulk-import fills from a Fidelity Activity CSV
